# DiscrimEval — explicit split

Reproduction of the explicit-side analysis in Christian & Mazor (2026),
[*Self-Blinding and Counterfactual Self-Simulation Mitigate Biases and Sycophancy in LLMs*](https://arxiv.org/abs/2601.14553).

**Design.** 65 DiscrimEval decision questions × 4 races (white, Black, Hispanic, Asian) × 2 genders,
demographics stated *explicitly* ("a Black female accountant"). Each prompt is paired with a blinded
`removed_template` (they/them, no race or gender) that a fresh replica of the model answers.

**Conditions.** `baseline` (cue visible) · `blinded` (the replica's own answer) · `ignore` (told to ignore
the cue) · `imagine` (asked to answer as if it had not seen the cue) · `self_sim` (given a tool that runs any
prompt on a fresh instance of itself) · `self_blind` (handed the replica's blinded answer).

**Claim to reproduce.** Instruction-only mitigations (`ignore`, `imagine`) leave the demographic effects
largely intact, while access to a blinded replica (`self_sim`, `self_blind`) pulls decisions toward the
replica's and removes most of the effect.

In [ ]:
import sys, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, "..")
import analyze as A

SPLIT = "explicit"
MODEL = None   # e.g. "claude-opus-5"; None = every model with results

df = A.load_results()
df = df[df["split"] == SPLIT] if len(df) else df
if MODEL and len(df):
    df = df[df["model"] == MODEL]
pd.set_option("display.width", 160); pd.set_option("display.float_format", lambda x: f"{x:.3f}")
if df.empty:
    print("No results yet. Collect with, e.g.:\n"
          "  python3 run_discrimeval.py --split " + SPLIT + " --condition baseline\n"
          "  python3 run_discrimeval.py --split " + SPLIT + " --condition blinded\n"
          "  ... then ignore / imagine / self_sim / self_blind")
else:
    print(df.groupby(["model", "condition"]).size().rename("n").to_frame())

## 1. Data quality: did every reply parse to yes/no?

In [ ]:
A.parse_rate(df) if len(df) else None

## 2. P(yes) by demographic cell and condition

Rows are race × gender; the white-male row is the reference. `blinded` is constant across rows by construction (the replica never sees the cue), so any spread there is sampling noise and calibrates the eye for the other columns.

In [ ]:
A.yes_rates(df) if len(df) else None

In [ ]:
# percentage-point gap from the white-male cell
A.yes_rate_vs_baseline(df) if len(df) else None

## 3. Demographic effects: logistic regression

`logit P(yes) = a_question + b_race + b_gender`, fixed effect per question, cluster-robust SEs by question. Coefficients are log-odds vs white male.

In [ ]:
tidy = pd.concat([A.bias_by_condition(d).assign(model=m) for m, d in df.groupby('model')]) if len(df) else pd.DataFrame()
tidy

In [ ]:
tidy.groupby('model').apply(A.bias_index) if len(tidy) else None

In [ ]:
if len(tidy):
    for model, t in tidy.groupby('model'):
        A.plot_bias(t, title=f'{SPLIT} · {model}')
    plt.tight_layout(); plt.show()

## 4. Do the mitigations move decisions toward the blinded replica?

Agreement between each condition's decision and the `blinded` run's majority answer for the same question.

In [ ]:
A.agreement_with_blinded(df) if len(df) else None

## 5. Self-simulation: did the model actually use the tool, and follow it?

In [ ]:
A.self_sim_fidelity(df) if len(df) else None

## 6. Per-question view

Which decision questions carry the largest demographic gaps in the baseline condition?

In [ ]:
if len(df):
    b = df[df["condition"] == "baseline"].dropna(subset=["yes"])
    cell = b.groupby(["nickname", "race", "gender"])["yes"].mean().unstack(["race", "gender"])
    gap = (cell.max(axis=1) - cell.min(axis=1)).sort_values(ascending=False).rename("max_cell_gap")
    display(gap.head(15).to_frame())